# Amazon Bedrock AgentCore Runtime上でのMCPサーバーのホスティング - AWS IAMインバウンド認証

## 概要

このチュートリアルでは、Amazon Bedrock AgentCore Runtime上でMCP（Model Context Protocol）サーバーをホスティングする方法を学習します。Amazon Bedrock AgentCore Python SDKを使用して、MCPツールをAmazon Bedrock AgentCoreと互換性のあるMCPサーバーとしてラップします。

Amazon Bedrock AgentCore Python SDKはMCPサーバーの実装詳細を処理するため、ツールのコア機能に集中できます。コードをAgentCore標準化されたMCPプロトコルコントラクトに変換して、直接通信を可能にします。

[MCPプロトコル](https://modelcontextprotocol.io/docs/getting-started/intro)仕様は従来、認証にOAuthトークンを必要としますが、AgentCore runtimeでは、MCPサーバーへのインバウンドリクエストにAWS IAM認証情報を設定する機能を提供し、重要なエンタープライズ要件に対応しています。

### チュートリアルの詳細

| 情報               | 詳細                                                       |
|:-------------------|:-----------------------------------------------------------|
| チュートリアルタイプ | ツールのホスティング                                       |
| ツールタイプ       | MCPサーバー                                                |
| チュートリアル構成要素 | AgentCore Runtime上でのMCPサーバーのホスティング           |
| チュートリアル垂直領域 | クロス垂直領域                                             |
| 例の複雑さ         | 簡単                                                       |
| 使用SDK            | Amazon BedrockAgentCore Python SDKおよびMCP               |

### チュートリアルアーキテクチャ

このチュートリアルでは、MCPサーバーをAgentCore runtimeにデプロイする方法について説明します。

デモンストレーションの目的で、3つのツールを持つシンプルなMCPサーバーを使用します：`add_numbers`、`multiply_numbers`、`greet_user`

<div style="text-align:left">
    <img src="images/hosting_mcp_server.png" width="60%"/>
</div>

### チュートリアルの主な機能

* カスタムツールを使用したMCPサーバーの作成
* MCPサーバーのローカルテスト
* Amazon Bedrock AgentCore Runtime上でのMCPサーバーのホスティング
* 認証を使用したデプロイ済みMCPサーバーの呼び出し


## 前提条件

このチュートリアルを実行するには、以下が必要です：
* Python 3.10+
* AWS認証情報の設定
* Amazon Bedrock AgentCore SDK
* MCP（Model Context Protocol）ライブラリ
* 実行中のDockerデーモン

In [ ]:
!pip install --force-reinstall -U -r requirements.txt --quiet

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
from bedrock_agentcore_starter_toolkit.operations.runtime import destroy_bedrock_agentcore
from boto3.session import Session
from pathlib import Path
import os

In [ ]:
boto_session = Session()
region = boto_session.region_name

agentcore_control_client = boto_session.client("bedrock-agentcore-control", region_name=region)
ssm_client = boto_session.client('ssm', region_name=region)

tool_name = "mcp_server_iam"

## MCP（Model Context Protocol）の理解

MCPは、AIモデルが外部データとツールに安全にアクセスできるようにするプロトコルです。主要な概念：

* **ツール**: AIがアクションを実行するために呼び出すことができる関数
* **Streamable HTTP**: AgentCore Runtimeで使用されるトランスポートプロトコル
* **セッション分離**: 各クライアントは`Mcp-Session-Id`ヘッダーを介して分離されたセッションを取得
* **ステートレス操作**: サーバーはスケーラビリティのためにステートレス操作をサポートする必要があります

AgentCore Runtimeは、MCPサーバーがデフォルトパスとして`0.0.0.0:8000/mcp`でホスティングされることを期待します。

### プロジェクト構造

適切な構造でプロジェクトを設定しましょう：

```
mcp_server_project/
├── mcp_server.py              # メインMCPサーバーコード
├── mcp_client.py          # ローカルテストクライアント
├── mcp_client_remote.py   # リモートテストクライアント
├── requirements.txt          # 依存関係
└── __init__.py              # Pythonパッケージマーカー
```

## MCPサーバーの作成

3つのシンプルなツールを使用してMCPサーバーを作成しましょう。サーバーは`stateless_http=True`を使用したFastMCPを使用します。これはAgentCore Runtimeとの互換性に必要です。

In [ ]:
%%writefile mcp_server.py
from mcp.server.fastmcp import FastMCP
from starlette.responses import JSONResponse

mcp = FastMCP(host="0.0.0.0", stateless_http=True)

@mcp.tool()
def add_numbers(a: int, b: int) -> int:
    """2つの数値を足し合わせる"""
    return a + b

@mcp.tool()
def multiply_numbers(a: int, b: int) -> int:
    """2つの数値を掛け合わせる"""
    return a * b

@mcp.tool()
def greet_user(name: str) -> str:
    """名前でユーザーを挨拶する"""
    return f"Hello, {name}! Nice to meet you."

if __name__ == "__main__":
    mcp.run(transport="streamable-http")

### このコードの動作

* **FastMCP**: ツールをホスティングできるMCPサーバーを作成
* **@mcp.tool()**: Python関数をMCPツールに変換するデコレータ
* **stateless_http=True**: AgentCore Runtimeとの互換性に必要
* **ツール**: 異なるタイプの操作を示す3つのシンプルなツール

## ローカルテストクライアントの作成

AgentCore Runtimeにデプロイする前に、MCPサーバーをローカルでテストするクライアントを作成しましょう：

In [ ]:
%%writefile mcp_client.py
import asyncio

from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client

async def main():
    mcp_url = "http://localhost:8000/mcp"
    headers = {}

    async with streamablehttp_client(mcp_url, headers, timeout=120, terminate_on_close=False) as (
        read_stream,
        write_stream,
        _,
    ):
        async with ClientSession(read_stream, write_stream) as session:
            await session.initialize()
            tool_result = await session.list_tools()
            print("Available tools:")
            for tool in tool_result.tools:
                print(f"  - {tool.name}: {tool.description}")

if __name__ == "__main__":
    asyncio.run(main())

 ### ローカルでのテスト

MCPサーバーをローカルでテストするには：

1. **ターミナル1**: MCPサーバーを起動
   ```bash
   python mcp_server.py
   ```
   
2. **ターミナル2**: テストクライアントを実行
   ```bash
   python mcp_client.py
   ```

出力に3つのツールがリストされているはずです。

## AgentCore Runtimeデプロイメントの設定

次に、スターターキットを使用して、エントリーポイント、作成した実行ロール、およびrequirementsファイルを使用してAgentCore Runtimeデプロイメントを設定します。また、起動時にAmazon ECRリポジトリを自動作成するようにスターターキットを設定します。

設定ステップ中に、アプリケーションコードに基づいてDockerファイルが生成されます。

<div style="text-align:left">
    <img src="images/configure.png" width="60%"/>
</div>

In [ ]:
print(f"Using AWS region: {region}")

required_files = ["mcp_server.py", "requirements.txt"]
for file in required_files:
    if not os.path.exists(file):
        raise FileNotFoundError(f"Required file {file} not found")
print("All required files found ✓")

agentcore_runtime = Runtime()

print("Configuring AgentCore Runtime...")
response = agentcore_runtime.configure(
    entrypoint="mcp_server.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    protocol="MCP",
    agent_name=tool_name,
)
print("Configuration completed ✓")

## AgentCore RuntimeへのMCPサーバーの起動

Dockerファイルができたので、MCPサーバーをAgentCore Runtimeに起動しましょう。これにより、Amazon ECRリポジトリとAgentCore Runtimeが作成されます。

<div style="text-align:left">
    <img src="images/launch.png" width="85%"/>
</div>

In [ ]:
print("Launching MCP server to AgentCore Runtime...")
print("This may take several minutes...")
launch_result = agentcore_runtime.launch()
print("Launch completed ✓")
print(f"Agent ARN: {launch_result.agent_arn}")
print(f"Agent ID: {launch_result.agent_id}")

In [ ]:

agent_arn_response = ssm_client.put_parameter(
    Name='/mcp_server/runtime_iam/agent_arn',
    Value=launch_result.agent_arn,
    Type='String',
    Description='Agent ARN for MCP server with inbound auth',
    Overwrite=True
)
print("✓ Agent ARN stored in Parameter Store")

print("\nConfiguration stored successfully!")
print(f"Agent ARN: {launch_result.agent_arn}")

## リモートテストクライアントの作成

デプロイされたMCPサーバーをテストするクライアントを作成しましょう。このクライアントはAWSから必要な認証情報を取得し、デプロイされたサーバーに接続します：

In [ ]:
%%writefile mcp_client_remote.py       
import asyncio
import sys
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4


logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    AWS SigV4認証を使用したストリーミング可能なHTTPトランスポートを作成します。

    この関数は、AWS Signature Version 4（SigV4）を使用してリクエストを認証するMCPクライアントトランスポートを作成します。
    標準のMCPクライアントはネイティブにAWS IAM認証をサポートしていないため、この関数が必要です。

    Args:
        mcp_url (str): MCPゲートウェイエンドポイントのURL
        service_name (str): SigV4署名用のAWSサービス名（通常は"bedrock-agentcore"）
        region (str): ゲートウェイがデプロイされているAWSリージョン

    Returns:
        StreamableHTTPTransportWithSigV4: SigV4認証用に設定されたトランスポートインスタンス

    Example:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url=".../mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # 現在のboto3セッションからAWS認証情報を取得
    # これらの認証情報はSigV4でリクエストに署名するために使用されます
    session = boto3.Session()
    credentials = session.get_credentials()

    # SigV4署名機能を持つカスタムトランスポートを作成して返す
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    MCPクライアントからツールの完全なリストを取得し、ページネーションを処理します。

    MCPサーバーはページネーションされたレスポンスでツールを返す場合があります。この関数は
    ページネーションを自動的に処理し、利用可能なすべてのツールを単一のリストで返します。

    Args:
        client: MCPクライアントインスタンス（strands.tools.mcp.mcp_client.MCPClientから）

    Returns:
        list: MCPサーバーから利用可能なすべてのツールの完全なリスト

    Example:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # すべてのページを取得するまでループ
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # 取得するページが他にあるかチェック
        if tmp_tools.pagination_token is None:
            # これ以上のページはない - 完了
            more_tools = False
        else:
            # さらにページが存在する - 次のページを取得する準備
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    try:
        async with create_streamable_http_transport_sigv4(
            mcp_url=mcp_url, service_name="bedrock-agentcore", region=region
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}")
                    print(f"   Description: {tool.description}")
                    if hasattr(tool, "inputSchema") and tool.inputSchema:
                        properties = tool.inputSchema.get("properties", {})
                        if properties:
                            print(f"   Parameters: {list(properties.keys())}")
                    print()

                print(f"✅ Successfully connected to MCP server!")
                print(f"Found {len(tool_result.tools)} tools available.")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## デプロイされたMCPサーバーのテスト

リモートクライアントを使用してデプロイされたMCPサーバーをテストしましょう：

In [ ]:
print("Testing deployed MCP server...")
print("=" * 50)
!python mcp_client_remote.py

### リモートでのMCPツールの呼び出し

ツールをリストするだけでなく、呼び出してMCP機能全体を実証する拡張クライアントを作成しましょう：

In [ ]:
%%writefile invoke_mcp_tools.py
import asyncio
import sys
import os
import logging
import boto3
from boto3.session import Session
from mcp import ClientSession
from mcp.client.streamable_http import streamablehttp_client
from streamable_http_sigv4 import streamablehttp_client_with_sigv4

logging.basicConfig(
    level=logging.INFO, format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger(__name__)


def create_streamable_http_transport_sigv4(
    mcp_url: str, service_name: str, region: str
):
    """
    AWS SigV4認証を使用したストリーミング可能なHTTPトランスポートを作成します。

    この関数は、AWS Signature Version 4（SigV4）を使用してリクエストを認証するMCPクライアントトランスポートを作成します。
    標準のMCPクライアントはネイティブにAWS IAM認証をサポートしていないため、この関数が必要です。

    Args:
        mcp_url (str): MCPゲートウェイエンドポイントのURL
        service_name (str): SigV4署名用のAWSサービス名（通常は"bedrock-agentcore"）
        region (str): ゲートウェイがデプロイされているAWSリージョン

    Returns:
        StreamableHTTPTransportWithSigV4: SigV4認証用に設定されたトランスポートインスタンス

    Example:
        >>> transport = create_streamable_http_transport_sigv4(
        ...     mcp_url=".../mcp",
        ...     service_name="bedrock-agentcore",
        ...     region="us-west-2"
        ... )
    """
    # 現在のboto3セッションからAWS認証情報を取得
    # これらの認証情報はSigV4でリクエストに署名するために使用されます
    session = boto3.Session()
    credentials = session.get_credentials()

    # SigV4署名機能を持つカスタムトランスポートを作成して返す
    return streamablehttp_client_with_sigv4(
        url=mcp_url,
        credentials=credentials,
        service=service_name,
        region=region,
    )


def get_full_tools_list(client):
    """
    MCPクライアントからツールの完全なリストを取得し、ページネーションを処理します。

    MCPサーバーはページネーションされたレスポンスでツールを返す場合があります。この関数は
    ページネーションを自動的に処理し、利用可能なすべてのツールを単一のリストで返します。

    Args:
        client: MCPクライアントインスタンス（strands.tools.mcp.mcp_client.MCPClientから）

    Returns:
        list: MCPサーバーから利用可能なすべてのツールの完全なリスト

    Example:
        >>> mcp_client = MCPClient(lambda: create_transport())
        >>> all_tools = get_full_tools_list(mcp_client)
        >>> print(f"Found {len(all_tools)} tools")
    """
    more_tools = True
    tools = []
    pagination_token = None

    # すべてのページを取得するまでループ
    while more_tools:
        tmp_tools = client.list_tools_sync(pagination_token=pagination_token)

        tools.extend(tmp_tools)

        # 取得するページが他にあるかチェック
        if tmp_tools.pagination_token is None:
            # これ以上のページはない - 完了
            more_tools = False
        else:
            # さらにページが存在する - 次のページを取得する準備
            more_tools = True
            pagination_token = tmp_tools.pagination_token

    return tools


async def main():
    boto_session = Session()
    region = boto_session.region_name
    print(f"Using AWS region: {region}")

    ssm_client = boto3.client("ssm", region_name=region)

    agent_arn_response = ssm_client.get_parameter(
        Name="/mcp_server/runtime_iam/agent_arn"
    )
    agent_arn = agent_arn_response["Parameter"]["Value"]
    print(f"Retrieved Agent ARN: {agent_arn}")

    if not agent_arn:
        print("❌ Error: AGENT_ARN not found")
        sys.exit(1)

    encoded_arn = agent_arn.replace(":", "%3A").replace("/", "%2F")
    mcp_url = f"https://bedrock-agentcore.{region}.amazonaws.com/runtimes/{encoded_arn}/invocations?qualifier=DEFAULT"

    try:
        async with create_streamable_http_transport_sigv4(
                mcp_url=mcp_url, service_name="bedrock-agentcore", region=region
        ) as (
            read_stream,
            write_stream,
            _,
        ):
            async with ClientSession(read_stream, write_stream) as session:
                print("\n🔄 Initializing MCP session...")
                await session.initialize()
                print("✓ MCP session initialized")

                print("\n🔄 Listing available tools...")
                tool_result = await session.list_tools()

                print("\n📋 Available MCP Tools:")
                print("=" * 50)
                for tool in tool_result.tools:
                    print(f"🔧 {tool.name}: {tool.description}")

                print("\n🧪 Testing MCP Tools:")
                print("=" * 50)

                try:
                    print("\n➕ Testing add_numbers(5, 3)...")
                    add_result = await session.call_tool(
                        name="add_numbers", arguments={"a": 5, "b": 3}
                    )
                    print(f"   Result: {add_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n✖️  Testing multiply_numbers(4, 7)...")
                    multiply_result = await session.call_tool(
                        name="multiply_numbers", arguments={"a": 4, "b": 7}
                    )
                    print(f"   Result: {multiply_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                try:
                    print("\n👋 Testing greet_user('Alice')...")
                    greet_result = await session.call_tool(
                        name="greet_user", arguments={"name": "Alice"}
                    )
                    print(f"   Result: {greet_result.content[0].text}")
                except Exception as e:
                    print(f"   Error: {e}")

                print("\n✅ MCP tool testing completed!")

    except Exception as e:
        print(f"❌ Error connecting to MCP server: {e}")
        import traceback

        print("\n🔍 Full error traceback:")
        traceback.print_exc()
        sys.exit(1)


if __name__ == "__main__":
    asyncio.run(main())


## ツール呼び出しのテスト

実際にMCPツールを呼び出してテストしましょう：

In [ ]:
print("Testing MCP tool invocation...")
print("=" * 50)
!python invoke_mcp_tools.py

## 次のステップ

AgentCore RuntimeにMCPサーバーを正常にデプロイしたので、以下を実行できます：

1. **ツールの追加**: 追加のツールでMCPサーバーを拡張
2. **カスタム認証**: AWS IAMインバウンド認証を実装
3. **統合**: 他のAgentCoreサービスと統合

## クリーンアップ（オプション）

このチュートリアル中に作成されたリソースをクリーンアップする場合は、次のセルを実行してください：

In [ ]:
# try:
#     ssm_client.delete_parameter(Name='/mcp_server/runtime_iam/agent_arn')
#     print("✓ Parameter Store parameter deleted")
# except ssm_client.exceptions.ParameterNotFound:
#     print("ℹ️  Parameter Store parameter not found")

In [ ]:
# destroy_bedrock_agentcore(
#     config_path=Path(".bedrock_agentcore.yaml"),
#     agent_name=tool_name,
#     delete_ecr_repo=True
# )

# 🎉 おめでとうございます！

以下を正常に完了しました：

✅ **カスタムツールを使用してMCPサーバーを作成**  
✅ **MCPクライアントを使用してローカルでテスト**  
✅ **Amazon Cognitoで認証を設定**  
✅ **AgentCore Runtimeを使用してAWSにデプロイ**  
✅ **適切な認証でリモートから呼び出し**  
✅ **MCPの概念とベストプラクティスを学習**  

MCPサーバーは現在、Amazon Bedrock AgentCore Runtime上で実行されており、本番環境での使用準備が整っています！

## まとめ

このチュートリアルでは、以下を学習しました：
- FastMCPを使用したMCPサーバーの構築
- AgentCore互換性のためのステートレスHTTPトランスポートの設定
- AWS IAMインバウンド認証の設定
- AWS上でのMCPサーバーのデプロイと管理
- ローカルとリモートの両方でのテスト
- ツール呼び出しのためのMCPクライアントの使用

デプロイされたMCPサーバーは、より大きなAIアプリケーションとワークフローに統合できるようになりました！